# VizWiz answerability: full pipeline (Project 2.2, individual)

**Vishnu Vardhan Manikandan**, Multimodal Data Fusion, Adelaide University, Semester 2 2026

One run reproduces every result in the report:

1. **Data and features** (`src/features.py`): downloads VizWiz-VQA (all three splits, with the labelled test annotations released in April 2026), measures photo quality, and extracts frozen ViT-B/16, BERT-base and CLIP ViT-B/32 features.
2. **Training** (`src/train.py`): question-only, image-only, late, early, cross-attention and gated fusion on ViT/BERT features, plus early and gated fusion on CLIP features, 3 seeds each; then the design ablation (`configs/ablation.yaml`).
3. **Evaluation** (`src/evaluate.py`): test metrics over seeds, bootstrap confidence intervals, McNemar tests, calibration, subgroups, gate analysis, modality ablation, latency, failure cases and data quality.

Settings live in `configs/`, and formats are described in `docs/data_contract.md`. Thresholds, the late-fusion weight and early stopping all use the development split (official validation); the test split is only scored.

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all, and allow Google Drive access. It takes about 75 minutes, mostly the download and feature extraction. Features stay on the Colab disk; runs and results (small) are saved to `MyDrive/vizwiz_individual/`.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/vishnuvardhan0ffl/vizwiz-fusion-individual.git"  # @param {type:"string"}
RESULTS = "/content/drive/MyDrive/vizwiz_individual/results"

import os
from google.colab import drive
drive.mount("/content/drive")
REPO = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if not os.path.exists(REPO):
    !git clone -q {REPO_URL} {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# quick checks before the long run: training (synthetic data) and the full pipeline (stub encoders)
!python -m tests.test_smoke
!python -m tests.test_pipeline

## 2. Data and features

Downloads about 15 GB of photos, then extracts all features in one pass over the photos (about 20 minutes on a T4).

In [ ]:
!python -m src.features --data-dir /content/data --out /content/features

## 3. Training

All models share one training loop and `configs/default.yaml`, 3 seeds each. The ablation varies modality dropout and cross-attention depth (`configs/ablation.yaml`).

In [ ]:
!python -m src.train --config configs/default.yaml --model all --skip-existing

In [ ]:
for m in ["gated_md0", "gated_md30", "gated_clip_md0", "cross_attention_l2"]:
    !python -m src.train --config configs/ablation.yaml --model {m} --skip-existing

## 4. Evaluation

In [ ]:
!python -m src.evaluate --config configs/default.yaml --ablation-config configs/ablation.yaml --out {RESULTS}

In [ ]:
from IPython.display import Image, display
for name in ["data_quality", "model_comparison", "roc_pr_calibration", "subgroups_sharpness",
             "learning_curves", "modality_ablation", "failure_cases"]:
    path = f"{RESULTS}/figures/{name}.png"
    if os.path.exists(path):
        print(name)
        display(Image(path))

In [ ]:
!python scripts/config_table.py --out {RESULTS}/tables/appendix_a_configs.csv

## References

Brier, GW 1950, 'Verification of forecasts expressed in terms of probability', *Monthly Weather Review*, vol. 78, no. 1, pp. 1-3.

Devlin, J, Chang, M-W, Lee, K & Toutanova, K 2019, 'BERT: pre-training of deep bidirectional transformers for language understanding', in *Proceedings of the 2019 conference of the North American chapter of the Association for Computational Linguistics: human language technologies*, vol. 1, Association for Computational Linguistics, Minneapolis, USA, pp. 4171-4186.

Dietterich, TG 1998, 'Approximate statistical tests for comparing supervised classification learning algorithms', *Neural Computation*, vol. 10, no. 7, pp. 1895-1923.

Dosovitskiy, A, Beyer, L, Kolesnikov, A, Weissenborn, D, Zhai, X, Unterthiner, T, Dehghani, M, Minderer, M, Heigold, G, Gelly, S, Uszkoreit, J & Houlsby, N 2021, 'An image is worth 16x16 words: transformers for image recognition at scale', in *International conference on learning representations (ICLR 2021)*, online.

Efron, B & Tibshirani, RJ 1993, *An introduction to the bootstrap*, Chapman & Hall, New York.

Gurari, D, Li, Q, Stangl, AJ, Guo, A, Lin, C, Grauman, K, Luo, J & Bigham, JP 2018, 'VizWiz grand challenge: answering visual questions from blind people', in *Proceedings of the IEEE conference on computer vision and pattern recognition (CVPR)*, IEEE, Salt Lake City, USA, pp. 3608-3617.

Guo, C, Pleiss, G, Sun, Y & Weinberger, KQ 2017, 'On calibration of modern neural networks', in *Proceedings of the 34th international conference on machine learning*, PMLR, vol. 70, pp. 1321-1330.

Holm, S 1979, 'A simple sequentially rejective multiple test procedure', *Scandinavian Journal of Statistics*, vol. 6, no. 2, pp. 65-70.

Mann, HB & Whitney, DR 1947, 'On a test of whether one of two random variables is stochastically larger than the other', *Annals of Mathematical Statistics*, vol. 18, no. 1, pp. 50-60.

McNemar, Q 1947, 'Note on the sampling error of the difference between correlated proportions or percentages', *Psychometrika*, vol. 12, no. 2, pp. 153-157.

Pech-Pacheco, JL, Cristobal, G, Chamorro-Martinez, J & Fernandez-Valdivia, J 2000, 'Diatom autofocusing in brightfield microscopy: a comparative study', in *Proceedings of the 15th international conference on pattern recognition (ICPR 2000)*, vol. 3, IEEE, Barcelona, Spain, pp. 314-317.

Radford, A, Kim, JW, Hallacy, C, Ramesh, A, Goh, G, Agarwal, S, Sastry, G, Askell, A, Mishkin, P, Clark, J, Krueger, G & Sutskever, I 2021, 'Learning transferable visual models from natural language supervision', in *Proceedings of the 38th international conference on machine learning (ICML 2021)*, PMLR, vol. 139, pp. 8748-8763.

Saito, T & Rehmsmeier, M 2015, 'The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets', *PLoS ONE*, vol. 10, no. 3, article no. e0118432.

VizWiz n.d., *Visual question answering*, VizWiz, viewed 20 October 2026, <https://vizwiz.org/tasks-and-datasets/vqa/>.

Wolf, T, Debut, L, Sanh, V, Chaumond, J, Delangue, C, Moi, A, Cistac, P, Rault, T, Louf, R, Funtowicz, M, Davison, J, Shleifer, S, von Platen, P, Ma, C, Jernite, Y, Plu, J, Xu, C, Le Scao, T, Gugger, S, Drame, M, Lhoest, Q & Rush, AM 2020, 'Transformers: state-of-the-art natural language processing', in *Proceedings of the 2020 conference on empirical methods in natural language processing: system demonstrations*, Association for Computational Linguistics, online, pp. 38-45.